# **Modelo VAR para `inflation_target`**

O modelo *Vector Autoregression* (VAR) é um modelo econométrico multivariado que prevê cada variável com base nos seus próprios valores passados e nos das restantes variáveis do sistema. É aplicado à previsão da inflação portuguesa para três subconjuntos de variáveis:

- **Teóricas** — selecionadas com base em fundamentos económicos.
- **Computacionais** — selecionadas por critérios estatísticos implementados.
- **Best Features** — variáveis em nível identificadas como mais importantes pelos modelos de *machine learning* via *feature importance*. Usam-se apenas variáveis em nível (sem *lags* pré-calculados) porque o VAR determina internamente o desfasamento, aplicando o mesmo *lag* a todas - garantindo comparabilidade entre subconjuntos.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import optuna
from IPython.display import display, HTML
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.api import VAR
from pipeline_datapreparation import prepare_dataset
import plotly.graph_objects as go

optuna.logging.set_verbosity(optuna.logging.WARNING)

TARGET = "inflation_target"

THEORETICAL = [
    "epu_pt_epu", "ULCIN_PT_ea-qd", "EXPGS_PT_ea-qd", "IMPGS_PT_ea-qd",
    "CCI_PT_ea-md", "GDP_PT_ea-qd", "UNETOT_PT_ea-md", "PPIPT_ppi",
]
COMPUTATIONAL = [
    "GS1_fred-md", "GS5_fred-md", "OILPRICEx_fred-md_Eur",
    "CUSR0000SAC_fred-md", "PCEPI_fred-md", "EMPENT_PT_ea-qd", "REER42_PT_ea-md",
    "HICPOV_PT_ea-md", "HICPNEF_PT_ea-md", "HICPSV_PT_ea-md",
    "HICPNG_PT_ea-md", "DFGDP_PT_ea-qd", "CCONFIX_PT_ea-md",
]

# Variáveis (em nível) mais importantes selecionadas pelo melhor modelo de ML
BEST_FEATURES = [
    "PCEPI_fred-md", "HICPOV_PT_ea-md", "HICPNEF_PT_ea-md",
    "HICPNG_PT_ea-md", "HICPSV_PT_ea-md", "PPIPT_ppi",
    "CCI_PT_ea-md", "EXPGS_PT_ea-qd", "epu_pt_epu",
]

SUBCONJUNTOS = [
    ("Teoricas",       THEORETICAL),
    ("Computacionais", COMPUTATIONAL),
    ("Best Features",  BEST_FEATURES),
]

HORIZONTES            = [3, 6, 12, 24]
OVERFITTING_THRESHOLD = 15.0
N_TRIALS              = 300
CORES = {
    "Teoricas":       "#5B8CC0",
    "Computacionais": "#F28E2B",
    "Best Features":  "#54A24B",
}

c:\Users\sasah\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **1. Carregar Dados**

São carregados os conjuntos de treino e teste já processados pelo *pipeline* de preparação de dados (séries estacionarizadas e variáveis explicativas normalizadas). O `CompactedData.csv` fornece os valores originais da inflação para reverter a diferenciação.

In [2]:
train = prepare_dataset("dados/train.csv", select_features=True, stationary=True, standardization=True, treat_outliers=False, create_lags=False, fit_scaler=True)["prepared_data"]
test  = prepare_dataset("dados/test.csv",  select_features=True, stationary=True, standardization=True, treat_outliers=False, create_lags=False, fit_scaler=False)["prepared_data"].iloc[12:]

ALL_VARS = sorted({v for _, vs in SUBCONJUNTOS for v in vs})
cols  = [TARGET] + ALL_VARS
train = train[cols].apply(pd.to_numeric, errors="coerce").dropna().asfreq("MS")
test  = test[cols].apply(pd.to_numeric, errors="coerce").dropna().asfreq("MS")

compact = pd.read_csv("dados/CompactedData.csv")
compact["Date"] = pd.to_datetime(compact["Date"], format="%Y-%m-%d")
orig = compact.set_index("Date")[TARGET]

print(f"Treino: {train.shape[0]} obs | {train.index.min().date()} a {train.index.max().date()}")
print(f"Teste:  {test.shape[0]} obs  | {test.index.min().date()} a {test.index.max().date()}")
print(f"Ancora teste (Jun/2023): {orig.loc[orig.index <= train.index[-1]].iloc[-1]}")

Treino: 280 obs | 2000-03-01 a 2023-06-01
Teste:  28 obs  | 2023-07-01 a 2025-10-01
Ancora teste (Jun/2023): 3.4


## **2. Funções Auxiliares**

- **`reverse_diff`** — reverte a diferenciação de primeira ordem a partir de um valor âncora.
- **`calcular_metricas`** — calcula MAE, RMSE e WAPE na escala original.
- **`verificar_overfitting`** — sinaliza *overfitting* quando o erro de teste supera o de treino em mais de 15%.
- **`optimizar_var_optuna`** — optimiza *lag* e *trend* com *Optuna* usando *walk-forward cross-validation* (3 *folds*) exclusivamente sobre os dados de treino. O *lag* máximo é limitado adaptativamente para evitar sobreparametrização.

In [3]:
def reverse_diff(series_diff, anchor_value):
    vals      = series_diff.values
    result    = np.empty(len(vals))
    result[0] = anchor_value + vals[0]
    for i in range(1, len(vals)):
        result[i] = result[i - 1] + vals[i]
    return pd.Series(result, index=series_diff.index, name=series_diff.name)


def calcular_metricas(y_real, y_pred, nome):
    y_real, y_pred = y_real.align(y_pred, join="inner")
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae  = mean_absolute_error(y_real, y_pred)
    wape = np.sum(np.abs(y_real - y_pred)) / np.sum(np.abs(y_real))
    return {"amostra": nome, "MAE": round(mae, 4), "RMSE": round(rmse, 4), "WAPE": round(wape, 4)}


def verificar_overfitting(m_treino, m_teste):
    rows = []
    for m in ["MAE", "RMSE", "WAPE"]:
        t, e    = m_treino[m], m_teste[m]
        aumento = round(((e - t) / t) * 100, 1) if t != 0 else np.nan
        rows.append({
            "metrica": m, "treino": t, "teste": e,
            "aumento_%": aumento,
            "overfitting": (aumento > OVERFITTING_THRESHOLD) if pd.notna(aumento) else False,
        })
    return pd.DataFrame(rows)


def optimizar_var_optuna(train_var, n_trials=N_TRIALS, n_folds=3, horizon=12):
    """
    Walk-forward cross-validation com limite adaptativo de lags.
    - n_folds janelas temporais expansiveis dentro do treino
    - max_lag calculado para garantir >= 3 obs por parametro por equacao
    - Objectivo: RMSE medio nos n_folds periodos de validacao
    """
    n          = len(train_var)
    k          = train_var.shape[1]
    # Tamanho do treino na 1ª fold; o maior entre o mínimo estatístico (k×3+1) e 55% dos dados
    min_train  = max(k * 3 + 1, int(n * 0.55))
    # A cada fold, o corte avanca fold_step obs; o horizonte de validacao é fixo
    fold_step  = (n - min_train - horizon) // n_folds
    # Limite de lag ("p": o nº do lag selecionado): cada equacao tem k*p+1 parametros; exige >= 3 observacoes por parametro
    max_lag_k  = max(1, min(12, min_train // (3 * k)))
    target_idx = train_var.columns.get_loc(TARGET)

    def objective(trial):
        """Funcao objetivo para o Optuna: RMSE medio nos n_folds periodos de validacao"""
        p     = trial.suggest_int("lag", 1, max_lag_k)
        trend = trial.suggest_categorical("trend", ["nc", "c", "ct"])
        rmses = []
        for i in range(n_folds):
            cutoff  = min_train + i * fold_step
            val_end = min(cutoff + horizon, n)
            tr  = train_var.iloc[:cutoff]
            val = train_var.iloc[cutoff:val_end]
            if len(val) == 0 or p >= len(tr):
                continue
            try:
                res = VAR(tr).fit(p, trend=trend)
                if not res.is_stable(verbose=False):
                    return float("inf")
                fc  = res.forecast(y=tr.values[-p:], steps=len(val))
                y_p = fc[:, target_idx]
                y_r = val[TARGET].values
                rmses.append(np.sqrt(mean_squared_error(y_r, y_p)))
            except Exception:
                return float("inf")
        return np.mean(rmses) if rmses else float("inf")

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    best["max_lag_k"] = max_lag_k
    return best

## **3. Optimização com Optuna**

O *Optuna* pesquisa a combinação óptima de *lag* (limitado adaptativamente por número de variáveis) e *trend* (`nc`, `c`, `ct`) para cada subconjunto. A optimização é feita exclusivamente nos dados de treino através de *walk-forward cross-validation* com 3 *folds* temporais expansíveis e horizonte de 12 meses por *fold*.

In [4]:
best_params = {}

for nome, variaveis in SUBCONJUNTOS:
    train_var         = train[[TARGET] + variaveis]
    params            = optimizar_var_optuna(train_var)
    best_params[nome] = params
    print(f"{nome}: lag={params['lag']}, trend='{params['trend']}'  "
          f"(max_lag permitido={params['max_lag_k']})")

Teoricas: lag=1, trend='ct'  (max_lag permitido=5)
Computacionais: lag=3, trend='ct'  (max_lag permitido=3)
Best Features: lag=2, trend='ct'  (max_lag permitido=5)


## **4. Treino do VAR**

Ajuste do modelo VAR final sobre a totalidade dos dados de treino com os hiperparâmetros óptimos. É verificada a estabilidade do modelo — condição necessária para que as previsões não divirjam.

In [5]:
modelos = {}

for nome, variaveis in SUBCONJUNTOS:
    train_var     = train[[TARGET] + variaveis]
    p             = best_params[nome]["lag"]
    t             = best_params[nome]["trend"]
    res           = VAR(train_var).fit(p, trend=t)
    modelos[nome] = (res, variaveis)
    print(f"{nome}: lag={p}, trend='{t}' | AIC={res.aic:.3f} | estavel={res.is_stable(verbose=False)}")

Teoricas: lag=1, trend='ct' | AIC=-15.096 | estavel=True
Computacionais: lag=3, trend='ct' | AIC=-19.595 | estavel=True
Best Features: lag=2, trend='ct' | AIC=-17.787 | estavel=True


## **5. Previsão e Reversão de Escala**

A previsão de teste é *multi-step ahead* — o modelo prevê os 28 meses de teste de uma só vez sem acesso a valores reais intermédios. Como a variável alvo foi diferenciada, reverte-se à escala original usando um valor âncora: o último valor real de treino (Jun/2023 = 3,4%).

In [6]:
anchor_teste = orig.loc[orig.index <= train.index[-1]].iloc[-1]
previsoes    = {}

for nome, (res, variaveis) in modelos.items():
    train_var = train[[TARGET] + variaveis]
    test_var  = test[[TARGET]  + variaveis]

    # Treino: ancora no valor original imediatamente anterior ao primeiro fitted value
    fitted        = res.fittedvalues[TARGET]
    anchor_treino = orig.loc[orig.index < fitted.index[0]].iloc[-1]
    real_treino_rev = reverse_diff(train_var[TARGET].loc[fitted.index], anchor_treino)
    pred_treino_rev = reverse_diff(fitted, anchor_treino)

    # Teste: ancora no ultimo valor original do treino
    forecast_arr = res.forecast(y=train_var.values[-res.k_ar:], steps=len(test_var))
    forecast_df  = pd.DataFrame(forecast_arr, index=test_var.index, columns=train_var.columns)
    real_teste_rev = reverse_diff(test_var[TARGET], anchor_teste)
    pred_teste_rev = reverse_diff(forecast_df[TARGET], anchor_teste)

    previsoes[nome] = {
        "real_treino": real_treino_rev, "pred_treino": pred_treino_rev,
        "real_teste":  real_teste_rev,  "pred_teste":  pred_teste_rev,
    }

print("Previsoes calculadas e revertidas para escala original.")

Previsoes calculadas e revertidas para escala original.


## **6. Métricas de Avaliação e Overfitting**

Métricas calculadas na escala original para treino (*fitted values*) e teste (*forecast*): MAE, RMSE e WAPE. Há *overfitting* se o erro de teste superar o de treino em mais de 15%.

In [7]:
overfitting_resultados = {}

for nome, p in previsoes.items():
    m_treino = calcular_metricas(p["real_treino"], p["pred_treino"], "treino")
    m_teste  = calcular_metricas(p["real_teste"],  p["pred_teste"],  "teste")
    tabela   = verificar_overfitting(m_treino, m_teste)
    overfitting_resultados[nome] = tabela
    print(f"\n-- {nome} --")
    display(tabela)


-- Teoricas --


,metrica,treino,teste,aumento_%,overfitting
0,MAE,1.1823,0.3948,-66.6,False
1,RMSE,1.5104,0.5098,-66.2,False
2,WAPE,0.5266,0.1621,-69.2,False



-- Computacionais --


,metrica,treino,teste,aumento_%,overfitting
0,MAE,1.0035,1.1521,14.8,False
1,RMSE,1.1884,1.2528,5.4,False
2,WAPE,0.4470,0.4730,5.8,False



-- Best Features --


,metrica,treino,teste,aumento_%,overfitting
0,MAE,1.0213,1.1813,15.7,True
1,RMSE,1.2424,1.4202,14.3,False
2,WAPE,0.4548,0.4850,6.6,False


## **7. Previsão por Horizonte**

Métricas calculadas cumulativamente para os primeiros 3, 6, 12 e 24 meses de previsão, permitindo analisar como o erro evolui com o horizonte temporal.

In [8]:
horizon_resultados = {}

for nome, p in previsoes.items():
    rows = []
    for h in HORIZONTES:
        r  = p["real_teste"].iloc[:h]
        pr = p["pred_teste"].iloc[:h]
        m  = calcular_metricas(r, pr, f"{h}m")
        rows.append({"horizonte": f"{h} meses", "MAE": m["MAE"], "RMSE": m["RMSE"], "WAPE": m["WAPE"]})
    horizon_resultados[nome] = pd.DataFrame(rows)

for metrica in ["MAE", "RMSE", "WAPE"]:
    fig = go.Figure()
    for nome, df_h in horizon_resultados.items():
        fig.add_trace(go.Bar(
            name=nome, x=df_h["horizonte"], y=df_h[metrica],
            marker_color=CORES[nome],
        ))
    fig.update_layout(
        title=f"VAR - {metrica} por horizonte de previsao",
        xaxis_title="Horizonte", yaxis_title=metrica,
        barmode="group", template="plotly_white",
        height=420, legend=dict(orientation="h", y=1.1),
    )
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

## **8. Previsão vs Real**

Comparação visual das previsões dos três subconjuntos com os valores reais da inflação no período de teste (Jul/2023 – Out/2025).

In [9]:
fig = go.Figure()

y_real = list(previsoes.values())[0]["real_teste"]
fig.add_trace(go.Scatter(
    x=y_real.index, y=y_real.values,
    name="Real", line=dict(color="#8E9091", width=2.5),
))

for nome, p in previsoes.items():
    fig.add_trace(go.Scatter(
        x=p["pred_teste"].index, y=p["pred_teste"].values,
        name=f"VAR ({nome})", line=dict(width=2, color=CORES[nome]),
    ))

fig.update_layout(
    title="Previsao VAR vs Real - inflation_target",
    xaxis_title="Data", yaxis_title="Inflacao (%)",
    template="plotly_white", height=500,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.05),
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

## **9. Guardar Melhor Modelo**

O melhor subconjunto é selecionado automaticamente com base no menor score médio (MAE + RMSE + WAPE) no conjunto de teste. O modelo VAR completo (treinado em treino + teste) é guardado em disco juntamente com o *scaler*, os parâmetros e os valores necessários para previsões futuras.

In [10]:
from sklearn.preprocessing import StandardScaler

os.makedirs("models", exist_ok=True)

# Selecionar automaticamente o melhor subconjunto (menor avg MAE+RMSE+WAPE no teste)
scores_teste  = {
    nome: df.set_index("metrica")["teste"][["MAE", "RMSE", "WAPE"]].mean()
    for nome, df in overfitting_resultados.items()
}
best_nome     = min(scores_teste, key=scores_teste.get)
variaveis_map = {nome: vars_ for nome, vars_ in SUBCONJUNTOS}
best_vars     = variaveis_map[best_nome]

p = best_params[best_nome]["lag"]
t = best_params[best_nome]["trend"]

print(f"Melhor subconjunto: {best_nome} (score={scores_teste[best_nome]:.4f})")

# Dataset completo (treino + teste) estacionarizado, sem standardização
raw_train = pd.read_csv("dados/train.csv", index_col="Date", parse_dates=True)
raw_test  = pd.read_csv("dados/test.csv",  index_col="Date", parse_dates=True)
raw_full  = pd.concat([raw_train, raw_test]).sort_index()
raw_full  = raw_full[~raw_full.index.duplicated(keep="last")]

stat_full = prepare_dataset(raw_full, stationary=True, standardization=False)["prepared_data"]
stat_full = stat_full[[TARGET] + best_vars].apply(pd.to_numeric, errors="coerce").dropna().asfreq("MS")

# Standardização sobre os dados completos (treino + teste)
scaler       = StandardScaler()
feature_cols = [c for c in stat_full.columns if c != TARGET]
stat_std     = stat_full.copy()
stat_std[feature_cols] = scaler.fit_transform(stat_full[feature_cols])

# VAR ajustado no dataset completo
model_full = VAR(stat_std).fit(p, trend=t)

# Âncora: último valor real de inflação no dataset completo
anchor_value = orig.loc[orig.index <= stat_std.index[-1]].iloc[-1]

joblib.dump({
    "model":        model_full,
    "scaler":       scaler,
    "variaveis":    best_vars,
    "params":       best_params[best_nome],
    "anchor_value": anchor_value,
    "last_y":       stat_std.values[-p:],
}, "models/VAR_BestModel.pkl")

print(f"Guardado: models/VAR_BestModel.pkl")
print(f"  Treinado em {stat_std.shape[0]} observações ({stat_std.index.min().date()} a {stat_std.index.max().date()})")
print(f"  lag={p}, trend='{t}'")
print(f"  Âncora: {anchor_value} ({stat_std.index[-1].date()})")

Melhor subconjunto: Teoricas (score=0.3556)
Guardado: models/VAR_BestModel.pkl
  Treinado em 309 observações (2000-02-01 a 2025-10-01)
  lag=1, trend='ct'
  Âncora: 2.3 (2025-10-01)


## **10. Previsão Futura**

Utilizando o modelo treinado no conjunto completo (treino + teste), são geradas previsões para os próximos 3 meses. A previsão é *multi-step ahead* e é revertida para a escala original usando o último valor real como âncora.

In [12]:
STEPS = 5

target_idx    = stat_std.columns.get_loc(TARGET)
forecast_arr  = model_full.forecast(y=stat_std.values[-p:], steps=STEPS)
forecast_diff = pd.Series(
    forecast_arr[:, target_idx],
    index=pd.date_range(stat_std.index[-1] + pd.DateOffset(months=1), periods=STEPS, freq="MS"),
)
forecast_orig = reverse_diff(forecast_diff, anchor_value)

# Histórico: últimos 24 meses dos dados reais
hist = orig.loc[orig.index <= stat_std.index[-1]].iloc[-24:]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hist.index, y=hist.values,
    name="Histórico", line=dict(color="#8E9091", width=2.5),
))
fig.add_trace(go.Scatter(
    x=[hist.index[-1], forecast_orig.index[0]],
    y=[hist.values[-1], forecast_orig.values[0]],
    line=dict(color=CORES[best_nome], width=2.5, dash="dash"),
    legendgroup="forecast", showlegend=False, hoverinfo="skip",
))
fig.add_trace(go.Scatter(
    x=forecast_orig.index, y=forecast_orig.values,
    name=f"Previsão VAR ({best_nome})",
    line=dict(color=CORES[best_nome], width=2.5, dash="dash"),
    mode="lines+markers", marker=dict(size=9),
    legendgroup="forecast",
))
fig.update_layout(
    title=f"VAR — Previsão {STEPS} meses futuros ({best_nome})",
    xaxis_title="Data", yaxis_title="Inflação (%)",
    template="plotly_white", height=450,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.05),
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

print(f"\nPrevisões para os próximos {STEPS} meses:")
for dt, val in zip(forecast_orig.index, forecast_orig.values):
    print(f"  {dt.strftime('%b/%Y')}: {val:.4f}%")


Previsões para os próximos 5 meses:
  Nov/2025: 2.2926%
  Dec/2025: 2.3036%
  Jan/2026: 2.3167%
  Feb/2026: 2.3303%
  Mar/2026: 2.3454%
